# Notebook 06 — Exploring Voyage AI embeddings

This notebook is a quick exploration of the Voyage AI embedding API before we
build the production pipeline. The goals are:

1. Verify the API key works
2. Understand the shape and properties of an embedding vector
3. Test semantic similarity manually with a few sample texts
4. Choose between `voyage-3-lite` and `voyage-3` for the project

We don't save anything from this notebook — it's pure exploration.

In [1]:
"""Notebook 06 — Explore Voyage AI embeddings."""

import os
from pathlib import Path

import numpy as np
import voyageai
from dotenv import load_dotenv

# Load API key from .env at project root
PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")

# Sanity check
print(f"Voyage API key loaded: {bool(os.environ.get('VOYAGE_API_KEY'))}")

# Client
vo = voyageai.Client()  # reads VOYAGE_API_KEY from environment
print("Voyage client ready")

Voyage API key loaded: True
Voyage client ready


In [2]:
# Try to embed a simple sentence
test_text = "A romantic candlelit bouchon in Vieux Lyon"

result = vo.embed(
    [test_text],
    model="voyage-3-lite",
    input_type="document",  # we're embedding a "document" (a restaurant text)
)

embedding = result.embeddings[0]
print(f"Model: voyage-3-lite")
print(f"Input: {test_text}")
print(f"Output: vector of length {len(embedding)}")
print(f"First 5 dimensions: {embedding[:5]}")
print(f"Last 5 dimensions: {embedding[-5:]}")
print(f"Tokens used: {result.total_tokens}")

Model: voyage-3-lite
Input: A romantic candlelit bouchon in Vieux Lyon
Output: vector of length 512
First 5 dimensions: [-0.030919212847948074, 0.04296404495835304, 0.04495082423090935, 0.013783263973891735, 0.007947106845676899]
Last 5 dimensions: [-0.06109338626265526, -0.0011796486796811223, -0.044702477753162384, -0.09635867178440094, -0.0009429428610019386]
Tokens used: 10


In [3]:
def cosine_similarity(a, b):
    """Cosine similarity between two vectors."""
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


# Reference
reference = "A romantic candlelit bouchon in Vieux Lyon"

# Candidates
candidates = [
    "Intimate dinner spot with cozy atmosphere",       # close to reference
    "Cheap fast food chain with bright lights",        # opposite
    "Traditional French restaurant with charm",        # somewhat close
    "Industrial Berlin techno club at 3 AM",           # completely unrelated
]

# Embed everything in one batch (more efficient than separate calls)
all_texts = [reference] + candidates
result = vo.embed(all_texts, model="voyage-3-lite", input_type="document")
embeddings = result.embeddings

# Similarities
ref_embedding = embeddings[0]
print(f"Reference: {reference!r}\n")
print(f"{'Similarity':>12} | Candidate")
print("-" * 80)
for i, cand in enumerate(candidates):
    sim = cosine_similarity(ref_embedding, embeddings[i + 1])
    print(f"{sim:>12.3f} | {cand}")

Reference: 'A romantic candlelit bouchon in Vieux Lyon'

  Similarity | Candidate
--------------------------------------------------------------------------------
       0.637 | Intimate dinner spot with cozy atmosphere
       0.553 | Cheap fast food chain with bright lights
       0.709 | Traditional French restaurant with charm
       0.501 | Industrial Berlin techno club at 3 AM


In [4]:
# Embedding the same text as document and as query
text = "romantic atmosphere"

doc_result = vo.embed([text], model="voyage-3-lite", input_type="document")
query_result = vo.embed([text], model="voyage-3-lite", input_type="query")

doc_emb = doc_result.embeddings[0]
query_emb = query_result.embeddings[0]

print(f"Document embedding[:5]: {doc_emb[:5]}")
print(f"Query embedding[:5]:    {query_emb[:5]}")
print(f"Similarity between the two: {cosine_similarity(doc_emb, query_emb):.4f}")

Document embedding[:5]: [-0.05839768052101135, 0.02072894759476185, 0.02552112378180027, 0.03588559851050377, -0.02685847505927086]
Query embedding[:5]:    [-0.08828850090503693, 0.03022921457886696, 0.023151740431785583, 0.02543092705309391, -0.05829919874668121]
Similarity between the two: 0.8329
